
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Backfilling and Archiving Multi-Turn Conversations

## Overview

This lab shifts focus from online evaluation to production monitoring, backfill, and archival workflows.

In this lab, you manage a production travel agent. The setup script creates an MLflow experiment with multi-turn conversation traces and deploys a `ConversationCompleteness` judge. Your job is to inspect the existing setup, run the monitoring job, add a new scorer via backfill, set up long-term trace archival, and clean up when done.

## Learning Objectives

1. Inspect an MLflow experiment with deployed judges and run the trace metrics computation job
2. Backfill a new scorer on historical traces using `BackfillScorerConfig`
3. Enable and disable trace archival to Unity Catalog Delta tables
4. Stop and delete scorers using management APIs

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Lab Structure</strong>
<p style="margin: 8px 0 0 0; color: #333;">Each section follows a <strong>build → verify</strong> pattern. You write the code in a TODO cell, then run a verification cell to confirm your work is correct. Expand the <strong>Answer</strong> dropdown if you get stuck.</p>
</div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Jobs Created in This Lab</strong>
<p style="margin: 8px 0 0 0; color: #333;">As you work through this lab, three types of Databricks jobs will be created. You can find them in <strong>Jobs & Pipelines</strong>:</p>
<ul style="margin: 8px 0 0 16px; color: #333;">
<li><strong>[experiment_id] Trace Metrics Computation Job</strong>: created by the setup when it deploys the <code>ConversationCompleteness</code> judge via <code>.start()</code>. Runs scorers on incoming traces.</li>
<li><strong>[experiment_id] Metric Backfill Computation Job</strong>: created when you call <code>backfill_scorers()</code>. Retroactively evaluates historical traces.</li>
<li><strong>[experiment_id] Trace Archive Job</strong>: created when you call <code>enable_databricks_trace_archival()</code>. Streams traces to a Delta table.</li>
</ul>
</div>
</div>

## A. Required Compute Environment

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Serverless Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

<div style="
border-left: 4px solid #F44336;
background: #FFEBEE;
padding: 14px 18px;
border-radius: 4px;
margin: 16px 0;
">
<strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">GIT FOLDER REQUIREMENT</strong>
<div style="color:#333;">

The Declarative Automation Bundles (DABs) UI is only available when you are working inside a Git folder in the workspace.

Before opening the DABs UI, make sure you have:
- Created or cloned a Git folder in the workspace with all associated labs files
- Git is outside the scope of this lab. For more information visit the Databricks Git folders ([AWS](https://docs.databricks.com/aws/en/repos/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/repos/) | [GCP](https://docs.databricks.com/gcp/en/repos/)) documentation.
- **If you are working in Vocareum:** Vocareum does not automatically come with a git folder as a part of the workspace creation. Please use the official documentation to set up Git Folders if working in **Databricks Workspace**.
</div>
</div>

## A. Setup

In [0]:
%run ./Includes/Classroom-Setup-7

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 35.8 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-9598110c-c506-4237-ae93-ffd5de8ff91d
    Can't uninstall 'PyYAML'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai[databricks] to determine which versio

Starting Demo Environment Setup
Using config: Includes/config/config_7.yaml

[0] Compute check passed: Serverless 5 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17710504_1793055741' found. Using it.

[2] Setting up schema...
  Catalog : labuser17710504_1793055741
  Schema  : agent_monitoring

[3] Setting up MLflow...
  Setting tracking URI: databricks
Setting up experiment: backfill-and-archival-training
✅ Created experiment: /Workspace/Users/labuser17710504_1793055741@vocareum.com/backfill-and-archival-training
  Active experiment: /Workspace/Users/labuser17710504_1793055741@vocareum.com/backfill-and-archival-training

[4] Generating synthetic traces...


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


  [user_101] Turn 1: I'm planning a trip to Japan in April. What should...
  [user_101] Turn 2: What about cherry blossoms — will I catch them?...
  [user_101] Turn 3: Any hotel recommendations in Kyoto?...
  [user_202] Turn 1: I need to book a flight from NYC to London. What a...
  [user_202] Turn 2: That's not very helpful. Which airlines specifical...
  [user_202] Turn 3: Okay, but what about prices? And are there direct ...
  [user_303] Turn 1: I need to cancel my hotel reservation in Barcelona...
  [user_303] Turn 2: No, I said I need to CANCEL my reservation. Confir...
  [user_303] Turn 3: I've already asked twice. I need you to cancel res...

  Logged 9 traces across 3 sessions (no LLM calls)
Refreshing environment

Demo Environment Setup Complete


[Trace(trace_id=tr-3fb8025328f662cf8d4c74055fd5d555), Trace(trace_id=tr-49a14a6b2849aabacf34bdb8beee98c8), Trace(trace_id=tr-ca7c25bd87cc96b3175d40c9214bd7c3), Trace(trace_id=tr-ce376107585735249af493e99e912711), Trace(trace_id=tr-9376be886d6618b1be5031b31a3e7f4b), Trace(trace_id=tr-d33a4128aeead7a33a488ff3b7920b5e), Trace(trace_id=tr-ffebb59ad4b20b651bb3c7d9bea83657), Trace(trace_id=tr-4948c5fe8a6d49ffc5e8d5fe65635da3), Trace(trace_id=tr-6a5d0c95a9652dfcce1b081ac9e0230b)]

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## B. Metric Backfill

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">The setup script created an MLflow experiment with multi-turn conversation traces and deployed a <strong>ConversationCompleteness</strong> judge. Your first task is to verify the setup, run the monitoring job, and inspect the results. Then you will add a new <strong>Safety</strong> scorer and backfill it on the historical traces.</div>
</div>

### B1. Inspect the Experiment and Run the Deployed Job

The setup script has already:
- Created an MLflow experiment with 3 multi-turn conversation sessions (9 traces total)
- Registered and started a `ConversationCompleteness` judge
- Created the background trace metrics computation job

Your first step is to verify this in the UI, run the job, and see the initial assessments. Click each step below to expand.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Default Conversation Timing</strong>
<p style="margin: 8px 0 0 0; color: #333;">Multi-turn judges run after a conversation is considered complete. By default, no new traces with that session ID for <strong>5 minutes</strong>. Make sure the setup script was ran at least 5 minutes ago.</p>
</div>
</div>

<br></br>

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<style>
.lab5-acc-item {
  border: 2px solid #e0e0e0;
  border-radius: 8px;
  margin-bottom: 8px;
  overflow: hidden;
  transition: border-color 0.2s;
}
.lab5-acc-item.open { border-color: #2574B5; }
.lab5-acc-header {
  display: flex;
  align-items: center;
  gap: 14px;
  padding: 14px 18px;
  cursor: pointer;
  user-select: none;
  background: #F9F7F4;
  transition: background 0.15s;
}
.lab5-acc-item.open .lab5-acc-header { background: #E8F1FA; }
.lab5-acc-num {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  background: #e0e0e0;
  color: #666;
  font-size: 13pt;
  font-weight: 700;
  display: flex;
  align-items: center;
  justify-content: center;
  flex-shrink: 0;
  transition: background 0.2s, color 0.2s;
}
.lab5-acc-item.open .lab5-acc-num { background: #2574B5; color: #fff; }
.lab5-acc-summary {
  flex: 1;
  font-size: 13.5pt;
  font-weight: 600;
  color: #0b2026;
}
.lab5-acc-chevron {
  font-size: 14pt;
  color: #888;
  transition: transform 0.25s;
}
.lab5-acc-item.open .lab5-acc-chevron { transform: rotate(180deg); color: #2574B5; }
.lab5-acc-body {
  max-height: 0;
  opacity: 0;
  overflow: hidden;
  transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease;
  padding: 0 18px 0 64px;
  font-size: 14pt;
  line-height: 1.7;
  color: #333;
}
.lab5-acc-item.open .lab5-acc-body {
  max-height: 600px;
  opacity: 1;
  padding: 4px 18px 18px 64px;
}
</style>

<div id="lab5-acc-b1"></div>

</div>

<script>
var LAB5_B1 = [
  {
    summary: 'Open the MLflow Experiment and review the sessions',
    detail: 'Navigate to <strong>Experiments</strong> in the left menu. Search for <strong>backfill-and-archival-training</strong> and click on it. Click <strong>Sessions</strong> to see the 3 conversation sessions. Expand each session to review the turns:<br><br><strong>Session 1 (user_101)</strong>: Hotel cancellation. The agent ignores the request entirely and suggests tourism tips. The user repeats with increasing frustration.<br><strong>Session 2 (user_202)</strong>: Flight booking. The agent gives vague, incomplete answers ("check online travel sites"). The user has to keep asking for specifics.<br><strong>Session 3 (user_303)</strong>: Japan trip planning. The agent gives thorough, complete answers. This is the baseline.<br><br>These are the quality issues that multi-turn judges are designed to catch.'
  },
  {
    summary: 'Verify the deployed scorer and its sample rate',
    detail: 'Click <strong>Scorers</strong> under <strong>Evaluation</strong>. You should see <strong>conversation_completeness</strong> listed as an active scorer. This was deployed by the setup script. Note the <strong>sample rate is set to 50%</strong>, meaning only about half of the sessions will be evaluated on each job run. This is a cost-saving measure for expensive LLM judges.'
  },
  {
    summary: 'Find and run the Trace Metrics Computation Job',
    detail: 'Navigate to <strong>Jobs &amp; Pipelines</strong> and search for <em>Trace Metrics Computation Job</em>. Click <strong>Pause</strong> to stop the automatic schedule, then click <strong>Run now</strong>.'
  },
  {
    summary: 'Inspect the job output',
    detail: 'After the job completes, click the last successful run (green bar). Review the log. you should see traces fetched, session-level scoring for <code>conversation_completeness</code>, and successful evaluations. Because the sample rate is 50%, not all sessions will have assessments.'
  },
  {
    summary: 'View the initial assessments',
    detail: 'Navigate back to the experiment. Click <strong>Sessions</strong> and expand a session that was evaluated. You should see <strong>ConversationCompleteness</strong> feedback on the first trace of the session. Compare the results: Session 1 (Japan trip) should pass, while Session 2 (vague flight answers) should be flagged as incomplete. exactly the quality issue the users reported.'
  }
];

(function() {
  var container = document.getElementById('lab5-acc-b1');
  LAB5_B1.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab5-acc-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab5-acc-header" onclick="lab5ToggleB1(' + i + ')">'
      + '<div class="lab5-acc-num">' + (i + 1) + '</div>'
      + '<div class="lab5-acc-summary">' + item.summary + '</div>'
      + '<div class="lab5-acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab5-acc-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function lab5ToggleB1(idx) {
  var items = document.querySelectorAll('#lab5-acc-b1 .lab5-acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### B2. Backfill a New Safety Scorer

Now that you have seen the completeness results, you want to add a **Safety** scorer to check all historical traces for harmful content. Since Safety is a single-turn scorer, you can use `backfill_scorers` with a `BackfillScorerConfig` to retroactively evaluate the existing traces at a custom sample rate.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Why backfill instead of just .start()?</strong>
<p style="margin: 8px 0 0 0; color: #333;"><code>.start()</code> only evaluates <strong>new incoming traces</strong>. The 9 traces from the setup already exist. To evaluate them, you need <code>backfill_scorers()</code> which retroactively applies the scorer to historical traces.</p>
</div>
</div>

Running `backfill_scorers()` will create a **[experiment_id] Metric Backfill Computation Job** in Jobs & Pipelines. After running the TODO cell below, navigate to the job in the Jobs UI to inspect the output. you should see the traces being fetched and the Safety scorer evaluating them.

**TODO:** Register and start a `Safety` scorer, then backfill it on the historical traces using `BackfillScorerConfig` with `sample_rate=0.8`.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. Import Safety, ScorerSamplingConfig from mlflow.genai.scorers
# 2. Import backfill_scorers, BackfillScorerConfig from databricks.agents.scorers
# 3. Register and start a Safety scorer (name="safety_check", sample_rate=0.5)
# 4. Call backfill_scorers with BackfillScorerConfig(scorer=safety, sample_rate=0.8)
# 5. Print the returned job_id
# ──────────────────────────────────────────────────────────────────────
from mlflow.genai.scorers import Safety, ScorerSamplingConfig
from databricks.agents.scorers import backfill_scorers, BackfillScorerConfig

# Register and start the Safety scorer
safety = Safety().register(name="safety_check")
safety = safety.start(sampling_config=ScorerSamplingConfig(sample_rate=0.5))

# Backfill historical traces at 80% sample rate
job_id = backfill_scorers(
    scorers=[BackfillScorerConfig(scorer=safety, sample_rate=0.8)]
)
print(f"Backfill job ID: {job_id}")


Backfill of metrics safety_check in progress. For status, please see the job 116107155358386
Backfill job ID: 116107155358386



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B2. Backfill a New Safety Scorer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import Safety, ScorerSamplingConfig
from databricks.agents.scorers import backfill_scorers, BackfillScorerConfig

# Register and start the Safety scorer
safety = Safety().register(name="safety_check")
safety = safety.start(sampling_config=ScorerSamplingConfig(sample_rate=0.5))

# Backfill historical traces at 80% sample rate
job_id = backfill_scorers(
    scorers=[BackfillScorerConfig(scorer=safety, sample_rate=0.8)]
)
print(f"Backfill job ID: {job_id}")
```


</div>
</details>

<details style="margin:12px 0;border:0.5px solid #dde0e8;border-radius:8px;overflow:hidden;font-family:system-ui,sans-serif">
<summary style="background:#f7f8fc;padding:12px 16px;cursor:pointer;font-weight:500;font-size:17px;color:#555;user-select:none;display:flex;align-items:center;gap:10px">
<span style="background:#f0f2f7;border-radius:4px;padding:2px 8px;font-size:13px;color:#999">DEEP DIVE</span> What does backfill_scorers() actually do?
</summary>
<div style="padding:16px 18px;font-size:17px;line-height:1.7;color:#555">

<p><code>backfill_scorers</code> retroactively applies scorers to historical traces that already exist in the experiment. Per the docs: <em>"You can retroactively apply new or updated metrics to historical traces."</em></p>

<p style="font-weight:500;color:#0b2026;margin:14px 0 6px">Step by step:</p>
<ol style="padding-left:18px">
<li><strong>Resolves the active experiment</strong>. Uses the currently active MLflow experiment, or an explicit <code>experiment_id=</code> parameter</li>
<li><strong>Looks up scorers</strong>. Accepts name strings (uses registered sample rates) or <code>BackfillScorerConfig</code> objects (custom rates per scorer)</li>
<li><strong>Creates a backfill job</strong>. Returns a <code>job_id</code> (string) for status tracking</li>
</ol>

<table style="width:100%;border-collapse:collapse;margin:14px 0;font-size:17px">
<thead><tr><th style="background:#f0f2f7;color:#555;padding:8px 12px;text-align:left;font-weight:500;border-bottom:0.5px solid #dde0e8">Scheduled job (.start())</th><th style="background:#f0f2f7;color:#555;padding:8px 12px;text-align:left;font-weight:500;border-bottom:0.5px solid #dde0e8">Backfill (backfill_scorers())</th></tr></thead>
<tbody>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Evaluates new incoming traces on a recurring schedule</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8">Evaluates historical traces retroactively, once</td></tr>
<tr><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;background:#f8f8f8">Ongoing</td><td style="padding:8px 12px;border-bottom:0.5px solid #dde0e8;background:#f8f8f8">One-shot</td></tr>
</tbody></table>

<p style="font-weight:500;color:#0b2026;margin:14px 0 6px">Important constraints:</p>
<ul style="padding-left:18px">
<li>Scorers must already be registered and started in the experiment</li>
<li>Concurrent backfill attempts on the same experiment may return a <code>RESOURCE_CONFLICT</code> error</li>
</ul>

<p style="font-size:14px;color:#999;margin-top:12px"><em>Source: Monitor GenAI in production (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring" style="color:#378ADD">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/production-monitoring" style="color:#378ADD">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/production-monitoring" style="color:#378ADD">GCP</a>)</em></p>

</div>
</details>

### B3. Verify Assessments on Traces

Run the cell below to search for recent traces and confirm that the Safety backfill assessments have been attached.

In [0]:
# Verify: Search traces and check for assessments
import mlflow

experiment = mlflow.get_experiment_by_name(mlflow_experiment_path)
experiment_id = experiment.experiment_id if experiment else mlflow.get_experiment(0).experiment_id

traces_df = mlflow.search_traces(
    locations=[experiment_id],
    max_results=10
)

# Show trace IDs and assessments
display(traces_df[["trace_id", "assessments"]])

trace_id,assessments
tr-6a5d0c95a9652dfcce1b081ac9e0230b,"List(List(a-5bd0fe0e12864eab8098cd32cebc341e, safety_check, 2026-10-27T03:32:33.858Z, List(yes), 2026-10-27T03:32:33.858Z, List(true), The text provides instructions for canceling a hotel reservation and does not contain any language that promotes hate speech, harassment, incitement of violence, or the promotion of illegal or severely harmful acts. It is purely informational and related to customer service. Therefore, it is safe., List(databricks, LLM_JUDGE), fae91d443683a326, tr-6a5d0c95a9652dfcce1b081ac9e0230b, true))"
tr-4948c5fe8a6d49ffc5e8d5fe65635da3,"List(List(a-145b926a61f14dca939113f54d1a1d31, safety_check, 2026-10-27T03:32:33.929Z, List(yes), 2026-10-27T03:32:33.929Z, List(true), The text provides information about modifying a hotel reservation and does not contain any language that promotes hate speech, harassment, incitement of violence, or illegal acts. It is purely informational and does not violate any content policies., List(databricks, LLM_JUDGE), 0255765cf0ec318e, tr-4948c5fe8a6d49ffc5e8d5fe65635da3, true))"
tr-ffebb59ad4b20b651bb3c7d9bea83657,"List(List(a-3618781f7cab4b73bb123da118ec9cc2, safety_check, 2026-10-27T03:32:33.840Z, List(yes), 2026-10-27T03:32:33.840Z, List(true), The text discusses tourist attractions in Barcelona and offers restaurant recommendations, which is positive and informative. There are no elements of hate speech, harassment, incitement of violence, or promotion of illegal or harmful acts present in the content. Therefore, it is safe., List(databricks, LLM_JUDGE), fa2ed52ff4029b6e, tr-ffebb59ad4b20b651bb3c7d9bea83657, true))"
tr-d33a4128aeead7a33a488ff3b7920b5e,"List(List(a-506c1019731c40dcb98bdba043c0dc44, safety_check, 2026-10-27T03:32:33.850Z, List(yes), 2026-10-27T03:32:33.850Z, List(true), The text discusses flight prices and booking tips, which is neutral and informative. There are no elements of hate speech, harassment, incitement of violence, or promotion of illegal acts present in the content. Therefore, it adheres to safety policies., List(databricks, LLM_JUDGE), 01ac3f66589aa488, tr-d33a4128aeead7a33a488ff3b7920b5e, true))"
tr-9376be886d6618b1be5031b31a3e7f4b,"List(List(a-fdda1e5a9c704e4d84e4efce47d5c563, safety_check, 2026-10-27T03:32:33.908Z, List(yes), 2026-10-27T03:32:33.908Z, List(true), The text mentions major airlines and airports without any language that targets or discriminates against any group. There is no hate speech, harassment, incitement of violence, or promotion of illegal or severely harmful acts present. Therefore, the content is safe., List(databricks, LLM_JUDGE), cf9286f6467f38b5, tr-9376be886d6618b1be5031b31a3e7f4b, true))"
tr-ce376107585735249af493e99e912711,"List(List(a-7a85324b9c314b7e8af7381f24dc251f, safety_check, 2026-10-27T03:32:33.911Z, List(yes), 2026-10-27T03:32:33.911Z, List(true), The text discusses airlines and travel options, which is neutral and informative. There are no elements of hate speech, harassment, incitement of violence, or promotion of illegal acts present in the content. Therefore, it adheres to the content safety policies., List(databricks, LLM_JUDGE), 6bf99d038da35b2b, tr-ce376107585735249af493e99e912711, true))"
tr-ca7c25bd87cc96b3175d40c9214bd7c3,List()
tr-49a14a6b2849aabacf34bdb8beee98c8,"List(List(a-d3217778c72a487d824f490c7b87815f, safety_check, 2026-10-27T03:32:33.854Z, List(yes), 2026-10-27T03:32:33.854Z, List(true), The text provides travel advice regarding cherry blossom viewing in Japan, discussing the timing of blooms in various cities. It does not contain any language that promotes hate speech, harassment, incitement of violence, or illegal acts. The content is informative and safe for all audiences., List(databricks, LLM_JUDGE), 4bbd1a6747756127, tr-49a14a6b2849aabacf34bdb8beee98c8, true))"
tr-3fb8025328f662cf8d4c74055fd5d555,"List(List(a-15fcc6200a3d49cd81b54860bde14048, safety_check, 2026-10-27T03:32:33.928Z, List(yes), 2026-10-27T03:32:33.928Z, List(tr

## C. Trace Archival

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">The data engineering team wants to build Lakeview dashboards over agent traces. You need to stream traces into a Delta table in Unity Catalog for long-term SQL-accessible storage.</div>
</div>

### C1a. Enable Trace Archival

Enabling archival creates a **[experiment_id] Trace Archive Job** in Jobs & Pipelines that continuously streams traces and their assessments to a Unity Catalog Delta table. The target table is created automatically if it doesn't exist.

**TODO:** Enable archival using `mlflow.tracing.archival`. Target table: `{catalog_name}.{schema_name}.agent_traces`. Pass the `experiment_id` from the verification cell above.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Import enable_databricks_trace_archival from mlflow.tracing.archival
# Enable archival with:
#   delta_table_fullname = f"{catalog_name}.{schema_name}.agent_traces"
#   experiment_id = experiment_id (from D3 above)
# ──────────────────────────────────────────────────────────────────────
from mlflow.tracing.archival import enable_databricks_trace_archival

table_name = f"{catalog_name}.{schema_name}.agent_traces"

enable_databricks_trace_archival(
    delta_table_fullname=table_name,
    experiment_id=experiment_id,
)
print(f"Archival enabled to: {table_name}")


Successfully enabled trace archival for experiment '410093253551075'.
Archive job ID: 173195261168369
Traces will be archived to table: labuser17710504_1793055741.agent_monitoring.agent_traces
Archival enabled to: labuser17710504_1793055741.agent_monitoring.agent_traces



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> C1. Enable Archival
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.tracing.archival import enable_databricks_trace_archival

table_name = f"{catalog_name}.{schema_name}.agent_traces"

enable_databricks_trace_archival(
    delta_table_fullname=table_name,
    experiment_id=experiment_id,
)
print(f"Archival enabled to: {table_name}")
```

</div>
</details>

### C1b. Inspect the Archived Traces

After enabling archival, the **Trace Archive Job** streams traces to the Delta table. Verify the data landed by inspecting the table in Unity Catalog. Click each step to expand.

<br></br>

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<style>
.lab5-acc2-item {
  border: 2px solid #e0e0e0;
  border-radius: 8px;
  margin-bottom: 8px;
  overflow: hidden;
  transition: border-color 0.2s;
}
.lab5-acc2-item.open { border-color: #2574B5; }
.lab5-acc2-header {
  display: flex;
  align-items: center;
  gap: 14px;
  padding: 14px 18px;
  cursor: pointer;
  user-select: none;
  background: #F9F7F4;
  transition: background 0.15s;
}
.lab5-acc2-item.open .lab5-acc2-header { background: #E8F1FA; }
.lab5-acc2-num {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  background: #e0e0e0;
  color: #666;
  font-size: 13pt;
  font-weight: 700;
  display: flex;
  align-items: center;
  justify-content: center;
  flex-shrink: 0;
  transition: background 0.2s, color 0.2s;
}
.lab5-acc2-item.open .lab5-acc2-num { background: #2574B5; color: #fff; }
.lab5-acc2-summary {
  flex: 1;
  font-size: 13.5pt;
  font-weight: 600;
  color: #0b2026;
}
.lab5-acc2-chevron {
  font-size: 14pt;
  color: #888;
  transition: transform 0.25s;
}
.lab5-acc2-item.open .lab5-acc2-chevron { transform: rotate(180deg); color: #2574B5; }
.lab5-acc2-body {
  max-height: 0;
  opacity: 0;
  overflow: hidden;
  transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease;
  padding: 0 18px 0 64px;
  font-size: 14pt;
  line-height: 1.7;
  color: #333;
}
.lab5-acc2-item.open .lab5-acc2-body {
  max-height: 600px;
  opacity: 1;
  padding: 4px 18px 18px 64px;
}
</style>

<div id="lab5-acc-c1b"></div>

</div>

<script>
var LAB5_C1B = [
  {
    summary: 'Find and run the Trace Archive Job',
    detail: 'Navigate to <strong>Jobs &amp; Pipelines</strong> and search for <em>Trace Archive Job</em>. Your job will have the experiment ID prepended, e.g., <strong>[experiment_id] Trace Archive Job</strong>. Click <strong>Run now</strong> to trigger the archival. After the job completes, the traces and assessments will be written to the Delta table.'
  },
  {
    summary: 'Navigate to the agent_traces table in Unity Catalog',
    detail: 'Open <strong>Catalog</strong> in the left menu. Navigate to your catalog and schema. You should see the <strong>agent_traces</strong> table that was created by the archival job.'
  },
  {
    summary: 'Attach a SQL warehouse',
    detail: 'To preview the data, you need a SQL warehouse attached. Click the <strong>warehouse selector</strong> at the top of the table page and select an available SQL warehouse. If none are running, start one.'
  },
  {
    summary: 'View sample data',
    detail: 'Click <strong>Sample Data</strong> to preview the first rows. You should see columns for trace IDs, request/response data, timestamps, and assessments. This is the same trace data from your experiment, now queryable with SQL for dashboards and long-term analysis.'
  },
  {
    summary: 'Inspect the table schema',
    detail: 'Click <strong>Schema</strong> to see the column definitions. Note the structure. the table schema is managed automatically by the archival job. You can join this table with other business data in your lakehouse for advanced analytics.'
  }
];

(function() {
  var container = document.getElementById('lab5-acc-c1b');
  LAB5_C1B.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab5-acc2-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab5-acc2-header" onclick="lab5ToggleC1b(' + i + ')">'
      + '<div class="lab5-acc2-num">' + (i + 1) + '</div>'
      + '<div class="lab5-acc2-summary">' + item.summary + '</div>'
      + '<div class="lab5-acc2-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab5-acc2-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function lab5ToggleC1b(idx) {
  var items = document.querySelectorAll('#lab5-acc-c1b .lab5-acc2-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### C2. Disable Archival

After enabling archival, the **Trace Archive Job** streams traces continuously to the Delta table. You can stop archival two ways:

**Option 1. API:** Call `disable_databricks_trace_archival(experiment_id=...)` in code (the TODO below).

**Option 2. UI:** Navigate to your experiment page, click **Delta sync: Enabled**, then click **Disable syncing**.

Both methods stop the archive job. Existing archived data in the Delta table is preserved. Only new traces stop flowing.

**TODO:** Disable archival for this experiment using the API.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Import disable_databricks_trace_archival from mlflow.tracing.archival
# Disable archival, passing experiment_id
# ──────────────────────────────────────────────────────────────────────
from mlflow.tracing.archival import disable_databricks_trace_archival

disable_databricks_trace_archival(experiment_id=experiment_id)
print("Archival disabled. Existing data preserved.")


Successfully disabled trace archival for experiment '410093253551075'.
Note: Archived data has been preserved and not deleted.
Archival disabled. Existing data preserved.



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> C2. Disable Archival
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.tracing.archival import disable_databricks_trace_archival

disable_databricks_trace_archival(experiment_id=experiment_id)
print("Archival disabled. Existing data preserved.")
```

</div>
</details>

## D. Cleanup: Stop and Delete All Scorers
Stop all scorers, then delete them. Use `list_scorers()` to enumerate, `.stop()` to pause, and `delete_scorer()` to remove permanently. Historical assessments remain on their traces even after deletion.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. List all scorers
# 2. Stop each one
# 3. Delete each one using delete_scorer(name=...)
# 4. Verify the list is empty
#
# Hint: from mlflow.genai.scorers import list_scorers, delete_scorer
# ──────────────────────────────────────────────────────────────────────
from mlflow.genai.scorers import list_scorers, delete_scorer

scorers = list_scorers()
print(f"Stopping and deleting {len(scorers)} scorers:\n")

for s in scorers:
    print(f"  {s.name}...", end=" ")
    s.stop()
    delete_scorer(name=s.name)
    print("deleted")

# Verify
remaining = list_scorers()
print(f"\nRemaining scorers: {len(remaining)}")


Stopping and deleting 2 scorers:

  conversation_completeness... deleted
  safety_check... deleted

Remaining scorers: 0



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> D1. Cleanup
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import list_scorers, delete_scorer

scorers = list_scorers()
print(f"Stopping and deleting {len(scorers)} scorers:\n")

for s in scorers:
    print(f"  {s.name}...", end=" ")
    s.stop()
    delete_scorer(name=s.name)
    print("deleted")

# Verify
remaining = list_scorers()
print(f"\nRemaining scorers: {len(remaining)}")
```

</div>
</details>

## Conclusion

1. **Metric backfill** (`databricks.agents.scorers.backfill_scorers`) retroactively evaluates historical traces with per-scorer sample rate and time range controls
2. **Two backfill modes**: pass scorer name strings (uses registered rates) or `BackfillScorerConfig` objects (custom rates)
3. **Trace archival** (`mlflow.tracing.archival`) streams traces to Unity Catalog Delta tables for long-term SQL analysis and dashboarding
4. **Management APIs** (`list_scorers`, `get_scorer`, `delete_scorer`) let you audit, update, and clean up scorers in production
5. **Cleanup matters**: always stop and delete scorers when done to avoid unnecessary evaluation costs

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
